In [2]:
# Cell 1: Environment Initialization
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Any, Tuple

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict

print("Stage 2 dependencies loaded successfully!")

Matplotlib is building the font cache; this may take a moment.


Stage 2 dependencies loaded successfully!


In [3]:
# Cell 2: Data Ingestion and ISO Physics Linkage
DATASET_PATH = 'data/SYNTHETIC_DATASET_100_CLEANED.csv'
df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns.")

def calculate_deterministic_gear_physics(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Stage 1 Core: Computes ISO derived mechanics upfront."""
    mn = float(inputs['mn'])
    z1 = int(inputs['z1'])
    z2 = int(inputs['z2'])
    beta_deg = float(inputs['beta'])
    alpha_deg = float(inputs.get('alpha', 20.0))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])
    
    beta_rad = np.radians(beta_deg)
    alpha_n_rad = np.radians(alpha_deg)
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    
    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    a = (d1 + d2) / 2.0
    v = (np.pi * d1 * n1) / 60000.0
    Ft = (2000.0 * T1) / d1
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    wt = Ft / (b * cos_beta_b)
    fm = (n1 * z1) / 60.0
    
    eps_beta = (b * np.sin(beta_rad)) / (np.pi * mn)
    eps_alpha = 1.88 - 3.2 * (1.0 / z1 + 1.0 / z2)
    eps_gamma = eps_alpha + eps_beta
    
    P = float(inputs.get('P', 0.0))
    if P <= 0.0:
        P = (T1 * n1) / 9550.0

    results = dict(inputs)
    results.update({
        'd1_mm': float(round(d1, 4)),
        'd2_mm': float(round(d2, 4)),
        'a': float(round(a, 3)),
        'v_mps': float(round(v, 3)),
        'Ft_N': float(round(Ft, 3)),
        'wt_N_per_mm': float(round(wt, 3)),
        'fm_Hz': float(round(fm, 2)),
        'eps_alpha': float(round(max(1.1, eps_alpha), 3)),
        'eps_beta': float(round(max(0.5, eps_beta), 3)),
        'eps_gamma': float(round(max(1.6, eps_gamma), 3)),
        'P': float(round(P, 2))
    })
    return results

print("Stage 1 physics pre-processor integrated into Stage 2 pipeline.")


Dataset loaded: 100 rows x 51 columns.
Stage 1 physics pre-processor integrated into Stage 2 pipeline.


In [4]:
# Cell 3: Forward Surrogates Training with 5-Fold Cross-Validation
ignore_cols = ['id', 'Sound', 'Mtrue', 'TEtrue', 'Mhat_v01', 'TEhat_v01', 'Soundhat_v01', 'Mat']
feature_cols = [c for c in df.columns if c not in ignore_cols]
X = df[feature_cols]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. Model M: Gradient Boosting Regressor
model_M = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
pred_m = cross_val_predict(model_M, X, df['Mtrue'], cv=cv)
r2_m = r2_score(df['Mtrue'], pred_m)
mae_m = mean_absolute_error(df['Mtrue'], pred_m)
model_M.fit(X, df['Mtrue'])

# 2. Model TE: L2 Regularized Ridge
model_TE = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=5.0))
])
pred_te = cross_val_predict(model_TE, X, df['TEtrue'], cv=cv)
r2_te = r2_score(df['TEtrue'], pred_te)
mae_te = mean_absolute_error(df['TEtrue'], pred_te)
model_TE.fit(X, df['TEtrue'])

# 3. Model Sound: L2 Regularized Ridge
model_Sound = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=5.0))
])
pred_s = cross_val_predict(model_Sound, X, df['Sound'], cv=cv)
r2_s = r2_score(df['Sound'], pred_s)
mae_s = mean_absolute_error(df['Sound'], pred_s)
model_Sound.fit(X, df['Sound'])

print("================ 5-FOLD CROSS-VALIDATION BENCHMARK RESULTS ================")
print(f"Misalignment (M):       R² = {r2_m:.3f} | MAE = {mae_m:.2f} µm  | Target Range: 20 - 100 µm")
print(f"Transmission Error (TE): R² = {r2_te:.3f} | MAE = {mae_te:.3f} µm | Target Range: 0.5 - 2.5 µm")
print(f"Sound Power Level (LW):  R² = {r2_s:.3f} | MAE = {mae_s:.2f} dB   | Target Range: 50 - 70 dB")
print("===========================================================================")

assert r2_m > 0.90, "Misalignment model failed to meet R² threshold (>0.90)"
assert r2_te > 0.75, "TE model failed to meet R² threshold (>0.75)"
assert r2_s > 0.80, "Sound model failed to meet R² threshold (>0.80)"
print("All 3 forward surrogates meet industrial predictive standards!")


================ 5-FOLD CROSS-VALIDATION BENCHMARK RESULTS ================
Misalignment (M):       R² = 0.951 | MAE = 4.35 µm  | Target Range: 20 - 100 µm
Transmission Error (TE): R² = 0.842 | MAE = 0.177 µm | Target Range: 0.5 - 2.5 µm
Sound Power Level (LW):  R² = 0.871 | MAE = 1.66 dB   | Target Range: 50 - 70 dB
All 3 forward surrogates meet industrial predictive standards!


In [5]:
# Cell 4: Range Validation & Forward Prediction Pipeline
def validate_output_ranges(m: float, te: float, sound: float) -> Dict[str, Any]:
    """Enforces the 3 industrial output range gates."""
    m_pass = (20.0 <= m <= 100.0)
    te_pass = (0.5 <= te <= 2.5)
    s_pass = (50.0 <= sound <= 70.0)
    all_pass = (m_pass and te_pass and s_pass)
    
    flags = []
    if not m_pass:
        flags.append(f"Misalignment ({m:.1f} µm) out of range [20, 100]")
    if not te_pass:
        flags.append(f"Transmission Error ({te:.2f} µm) out of range [0.5, 2.5]")
    if not s_pass:
        flags.append(f"Sound Power Level ({sound:.1f} dB) out of range [50, 70]")
        
    return {
        'Misalignment_Pass': m_pass,
        'TE_Pass': te_pass,
        'Sound_Pass': s_pass,
        'All_Pass': all_pass,
        'Status_Pill': 'PASS (ALL GATES CLEARED)' if all_pass else 'FAIL (RANGE VIOLATION)',
        'Flags': flags
    }

def evaluate_gear_nvh_forward(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Evaluates ISO physics, runs forward surrogates, and checks range gates."""
    full_inputs = calculate_deterministic_gear_physics(inputs)
    
    sample = df[feature_cols].iloc[0].copy()
    for k, v in full_inputs.items():
        if k in sample:
            sample[k] = v
            
    X_eval = pd.DataFrame([sample])
    
    m_val = float(model_M.predict(X_eval)[0])
    te_val = float(model_TE.predict(X_eval)[0])
    sound_val = float(model_Sound.predict(X_eval)[0])
    
    range_eval = validate_output_ranges(m_val, te_val, sound_val)
    
    return {
        'Outputs': {
            'Misalignment_um': round(m_val, 2),
            'Transmission_Error_um': round(te_val, 3),
            'Sound_Power_dB': round(sound_val, 2)
        },
        'Range_Validation': range_eval,
        'Operating_Physics': {
            'Pitch_Diameter_d1_mm': round(full_inputs['d1_mm'], 2),
            'Tangential_Force_Ft_N': round(full_inputs['Ft_N'], 2),
            'Specific_Load_wt_N_mm': round(full_inputs['wt_N_per_mm'], 2),
            'Mesh_Frequency_fm_Hz': round(full_inputs['fm_Hz'], 1),
            'Pitch_Velocity_v_mps': round(full_inputs['v_mps'], 2),
            'Total_Contact_Ratio': round(full_inputs['eps_gamma'], 2)
        }
    }

print("Forward evaluation pipeline and range validator ready.")

Forward evaluation pipeline and range validator ready.


In [6]:
# Cell 5: Automated Verification Test
test_deck_1 = {
    'mn': 2.0, 'z1': 18, 'z2': 118, 'beta': 22.0, 'alpha': 20.0,
    'b': 32.0, 'a': 146.7, 'x1': 0.13, 'x2': 0.05,
    'h_a': 2.0, 'h_f': 2.5, 'HRC': 59, 'HB': 315, 'CHD': 0.62,
    'σ_H0': 1484, 'σ_F0': 537, 'P': 55.6, 'n1': 12200, 'n2': 1861.0,
    'T1': 43.5, 'i': 6.56, 'K_A': 1.13, 'Kv': 1.323, 'KHbeta': 1.187, 'K_Hα': 1.059,
    'Q': 5, 'Ra': 0.81, 'Falpha': 3.8, 'Fbeta': 3.1, 'f_pt': 2.4,
    'Calpha': 10.0, 'Cbeta': 8.0, 'VG': 32, 'Toil': 66, 'λ': 1.9,
    'S_B': 1.8, 'S_H': 2.6, 'S_F': 2.8, 'L_h': 30000
}

report = evaluate_gear_nvh_forward(test_deck_1)

print("\n--- FORWARD EVALUATION BENCHMARK: CASE 1 ---")
print(f"Predicted Misalignment:     {report['Outputs']['Misalignment_um']} µm  (True: 31.7 µm)  -> Pass: {report['Range_Validation']['Misalignment_Pass']}")
print(f"Predicted Transmission Error: {report['Outputs']['Transmission_Error_um']} µm (True: 0.93 µm)  -> Pass: {report['Range_Validation']['TE_Pass']}")
print(f"Predicted Sound Power Level:  {report['Outputs']['Sound_Power_dB']} dB  (True: 51.9 dB)  -> Pass: {report['Range_Validation']['Sound_Pass']}")
print(f"Overall Validation Verdict:   {report['Range_Validation']['Status_Pill']}")
print(f"Operating Forces:            Ft = {report['Operating_Physics']['Tangential_Force_Ft_N']} N, wt = {report['Operating_Physics']['Specific_Load_wt_N_mm']} N/mm")

assert report['Range_Validation']['All_Pass'] is True, "Range validation failed on Case 1!"
print("\nVERDICT: STAGE 2 FORWARD EVALUATION PASSED. PROCEED TO STAGE 3.")



--- FORWARD EVALUATION BENCHMARK: CASE 1 ---
Predicted Misalignment:     31.34 µm  (True: 31.7 µm)  -> Pass: True
Predicted Transmission Error: 1.049 µm (True: 0.93 µm)  -> Pass: True
Predicted Sound Power Level:  54.25 dB  (True: 51.9 dB)  -> Pass: True
Overall Validation Verdict:   PASS (ALL GATES CLEARED)
Operating Forces:            Ft = 2240.69 N, wt = 76.24 N/mm

VERDICT: STAGE 2 FORWARD EVALUATION PASSED. PROCEED TO STAGE 3.
